In [39]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from typing import Dict
import warnings
warnings.filterwarnings('ignore')
import copy
from tqdm import tqdm
import random
import scanpy as sc
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch import nn

from stformer import logger
from stformer.tokenizer import GeneVocab
from stformer.tokenizer import tokenize_and_pad_batch_2
from stformer.model import TransformerModel

In [56]:
class Tokenizer():
    def __init__(self, tokenizer_dir, adata, vocab, pad_value, pad_token):
        self.tokenizer_dir = tokenizer_dir
        self.adata = adata
        self.vocab = vocab
        self.pad_value = pad_value
        self.pad_token = pad_token
        self.load_data()
    
    def load_data(self):
        self.expression_matrix = self.adata.X
        self.niche_ligands_expression = self.adata.obsm['niche_ligands_expression']
        self.niche_composition = self.adata.obsm['niche_composition']
        self.niche_celltypes = self.adata.obsm['niche_celltypes']
        self.cell_types_list = list(self.adata.uns['cell_types_list'])

        gene_list_df = pd.read_csv(f'{self.tokenizer_dir}/OS_scRNA_gene_index.19264.tsv', header=0, delimiter='\t')
        self.gene_list = list(gene_list_df['gene_name'])
        self.gene_ids = np.array(self.vocab(self.gene_list), dtype=int)

        ligand_database = pd.read_csv(self.tokenizer_dir+'ligand_database.csv', header=0, index_col=0)
        ligand_symbol = ligand_database[ligand_database.sum(1)>1].index.values
        self.ligand_symbol = gene_list_df.loc[gene_list_df['gene_name'].isin(ligand_symbol), 'gene_name'].values
        self.ligand_ids = np.array(self.vocab(self.ligand_symbol.tolist())*50, dtype=int)

    def perturb_data(self, pert_receptor=False, pert_ligand=False, pert_receptor_celltype=None, pert_ligand_celltype=None, pert_receptor_genes=None, pert_ligand_genes=None, pert_fold=1):
        if pert_receptor_celltype is not None:
            pert_receptor_celltype = self.cell_types_list.index(pert_receptor_celltype)+1
        
        if pert_ligand_celltype is not None:
            pert_ligand_celltype = self.cell_types_list.index(pert_ligand_celltype)+1

        pert_expression_matrix = []
        pert_niche_ligands_expression = []
        pert_niche_composition = []
        pert_celltypes = []

        sample_celltypes = self.adata.obs['cell_type'].values

        for i in range(self.adata.shape[0]):
            if pert_receptor_celltype is not None and sample_celltypes[i] != pert_receptor_celltype:
                continue
            
            if pert_ligand_celltype is not None and self.niche_composition[i, np.where(self.niche_celltypes[i].A == pert_ligand_celltype)[1]].sum() == 0:
                continue
            
            pert_expression_matrix_i = self.expression_matrix[i].A
            # if pert_expression_matrix_i[0, np.isin(self.gene_list, pert_receptor_genes)].sum() == 0:
            #     continue

            if pert_ligand_celltype is None:
                pert_ligand_cellindex = np.array(range(50))
            else:
                pert_ligand_cellindex = np.where(self.niche_celltypes[i].A == pert_ligand_celltype)[1]
            
            pert_niche_ligands_expression_i = self.niche_ligands_expression[i].A.reshape(50,986)
            if pert_niche_ligands_expression_i[pert_ligand_cellindex][:, np.isin(self.ligand_symbol, pert_ligand_genes)].sum() == 0:
                continue
            
            if pert_receptor:
                pert_expression_matrix_i[0, np.where((np.isin(self.gene_list, pert_receptor_genes) & (pert_expression_matrix_i[0] > 0))==True)[0][0]] = pert_expression_matrix_i[0, np.where((np.isin(self.gene_list, pert_receptor_genes) & (pert_expression_matrix_i[0] > 0))==True)[0][0]]*pert_fold
            if pert_ligand:
                for ci in pert_ligand_cellindex:
                    pert_niche_ligands_expression_i[ci, np.where(np.isin(self.ligand_symbol, pert_ligand_genes)==True)[0][0]] = pert_niche_ligands_expression_i[ci, np.where(np.isin(self.ligand_symbol, pert_ligand_genes)==True)[0][0]]*pert_fold
            
            pert_expression_matrix.append(pert_expression_matrix_i)
            pert_niche_ligands_expression.append(pert_niche_ligands_expression_i.reshape(1,-1))
            pert_niche_composition.append(self.niche_composition[i].A)
            pert_celltypes.append(sample_celltypes[i])

        self.pert_expression_matrix = np.concatenate(pert_expression_matrix)
        self.pert_niche_ligands_expression = np.concatenate(pert_niche_ligands_expression)
        self.pert_niche_composition = np.concatenate(pert_niche_composition)
        self.pert_celltypes = torch.tensor(pert_celltypes)

    def tokenize_data(self):

        biases = np.zeros([self.pert_niche_composition.shape[0], self.pert_niche_composition.shape[1]*986])
        for k in range(biases.shape[0]):
            biases[k] = np.concatenate([[np.log(p)]*986 for p in self.pert_niche_composition[k]])

        tokenized_data = tokenize_and_pad_batch_2(
            self.pert_expression_matrix,
            self.pert_niche_ligands_expression,
            biases,
            self.gene_ids,
            self.ligand_ids,
            pad_id = self.vocab[self.pad_token],
            pad_value = self.pad_value,
        )
        
        logger.info(
            f"number of samples: {tokenized_data['center_genes'].shape[0]}, "
            f"\n\t feature length of center cell: {tokenized_data['center_genes'].shape[1]}"
            f"\n\t feature length of niche cells: {tokenized_data['niche_genes'].shape[1]}"
        )

        self.tokenized_data = tokenized_data

    def prepare_data(self):
        self.data_pt = {
            "center_gene_ids": self.tokenized_data["center_genes"],
            "input_center_values": self.tokenized_data["center_values"],
            "niche_gene_ids": self.tokenized_data["niche_genes"],
            "input_niche_values": self.tokenized_data["niche_values"],
            "cross_attn_bias": self.tokenized_data["cross_attn_bias"],
            "celltype_labels": self.pert_celltypes,
        }

    def prepare_dataloader(self, batch_size):
        data_loader = DataLoader(
            dataset=SeqDataset(self.data_pt),
            batch_size=batch_size,
            shuffle=False,
            drop_last=False,
            num_workers=min(len(os.sched_getaffinity(0)), batch_size // 2),
            pin_memory=True,
        )
        return data_loader

class SeqDataset(Dataset):
    def __init__(self, data: Dict[str, torch.Tensor]):
        self.data = data

    def __len__(self):
        return self.data["center_gene_ids"].shape[0]

    def __getitem__(self, idx):
        return {k: v[idx] for k, v in self.data.items()}

In [57]:
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    """
    Evaluate the model on the evaluation data.
    """
    amp = True

    model.eval()
    cell_embeddings_list = []

    with torch.no_grad():
        for batch_data in tqdm(loader):
            center_gene_ids = batch_data["center_gene_ids"].to(device)
            if center_gene_ids.size(0)<12:
                continue
            input_center_values = batch_data["input_center_values"].to(device)
            niche_gene_ids = batch_data["niche_gene_ids"].to(device)
            input_niche_values = batch_data["input_niche_values"].to(device)
            cross_attn_bias = batch_data["cross_attn_bias"].to(device)

            encoder_src_key_padding_mask = niche_gene_ids.eq(vocab[pad_token])
            decoder_src_key_padding_mask = center_gene_ids.eq(vocab[pad_token])

            with torch.cuda.amp.autocast(enabled=amp):
                output_dict = model(
                        niche_gene_ids,
                        input_niche_values,
                        encoder_src_key_padding_mask,
                        center_gene_ids,
                        input_center_values,
                        decoder_src_key_padding_mask,
                        cross_attn_bias,
                    )

                cell_embeddings_list.append(output_dict["cell_emb"].to('cpu'))

    cell_embeddings = torch.cat(cell_embeddings_list)

    return cell_embeddings

In [58]:
from tasks.scfoundation import load

def initialize_model(model_file):
    pretrainmodel, pretrainconfig = load.load_model_frommmf('../../tasks/scfoundation/models/models.ckpt')

    model = TransformerModel(
            embsize,
            nhead,
            d_hid,
            nlayers,
            do_gcl = True,
            nlayers_gcl = 3,
            n_gcl =2,
            dropout = dropout,
            cell_emb_style = cell_emb_style,
            scfoundation_token_emb1 = copy.deepcopy(pretrainmodel.token_emb),
            scfoundation_token_emb2 = copy.deepcopy(pretrainmodel.token_emb),
            scfoundation_pos_emb1 = copy.deepcopy(pretrainmodel.pos_emb),
            scfoundation_pos_emb2 = copy.deepcopy(pretrainmodel.pos_emb),
        )

    pt_model = torch.load(model_file, map_location='cpu')

    model_dict = model.state_dict()
    pretrained_dict = pt_model.state_dict()
    pretrained_dict = {
                k: v
                for k, v in pretrained_dict.items()
                if 'cls_decoder' not in k and 'gcl_decoder' not in k
    }
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict)

    pre_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())
    for name, para in model.named_parameters():
        para.requires_grad = False
    post_freeze_param_count = sum(dict((p.data_ptr(), p.numel()) for p in model.parameters() if p.requires_grad).values())
    
    logger.info(f"Total Pre freeze Params {(pre_freeze_param_count )}")
    logger.info(f"Total Post freeze Params {(post_freeze_param_count )}")

    return model

In [101]:
embsize = 768
d_hid = 3072
nhead = 12
nlayers = 6
dropout = 0.1
cell_emb_style = 'max-pool'

pad_token = "<pad>"
pad_value = 103
tokenizer_dir = '../../stformer/tokenizer/'
vocab_file = tokenizer_dir + "scfoundation_gene_vocab.json"
vocab = GeneVocab.from_file(vocab_file)
vocab.append_token(pad_token)
vocab.set_default_index(vocab[pad_token])

batch_size = 20

In [102]:
model_file = '../../pretraining/models/model_4.1M.ckpt'

model = initialize_model(model_file)
model = nn.DataParallel(model, device_ids = [0])
device = torch.device("cuda:0")
model.to(device)

stFormer - INFO - Total Pre freeze Params 88849787
stFormer - INFO - Total Post freeze Params 0


DataParallel(
  (module): TransformerModel(
    (scfoundation_token_emb1): AutoDiscretizationEmbedding2(
      (mlp): Linear(in_features=1, out_features=100, bias=True)
      (mlp2): Linear(in_features=100, out_features=100, bias=True)
      (LeakyReLU): LeakyReLU(negative_slope=0.1)
      (Softmax): Softmax(dim=-1)
      (emb): Embedding(100, 768)
      (emb_mask): Embedding(1, 768)
      (emb_pad): Embedding(1, 768)
    )
    (scfoundation_token_emb2): AutoDiscretizationEmbedding2(
      (mlp): Linear(in_features=1, out_features=100, bias=True)
      (mlp2): Linear(in_features=100, out_features=100, bias=True)
      (LeakyReLU): LeakyReLU(negative_slope=0.1)
      (Softmax): Softmax(dim=-1)
      (emb): Embedding(100, 768)
      (emb_mask): Embedding(1, 768)
      (emb_pad): Embedding(1, 768)
    )
    (scfoundation_pos_emb1): Embedding(19267, 768)
    (scfoundation_pos_emb2): Embedding(19267, 768)
    (transformer_decoder): BiasedTransformerDecoder(
      (layers): ModuleList(
     

In [ ]:
adata = sc.read_h5ad('liver_cosmx_TME_niche.h5ad')
ct = 'CD3+.alpha.beta.T.cells'
adata = adata[adata.obs['cell_type']==list(adata.uns['cell_types_list']).index(ct)+1]

tokenizer = Tokenizer(tokenizer_dir, adata, vocab, pad_value, pad_token)

In [104]:
nonzero_ligand_genes = set(np.array((tokenizer.ligand_symbol.tolist())*50)[np.nonzero(adata.obsm['niche_ligands_expression'])[1]])

In [105]:
cell_embeddings_results = {}
for l in nonzero_ligand_genes:
    print(f"pert_ligand: {l}")

    tokenizer.perturb_data(False, True, None, None, None, l, 1)
    tokenizer.tokenize_data()
    tokenizer.prepare_data()
    data_loader = tokenizer.prepare_dataloader(batch_size)
    cell_embeddings_0 = evaluate(model, data_loader)

    tokenizer.perturb_data(False, True, None, None, None, l, 1/3)
    tokenizer.tokenize_data()
    tokenizer.prepare_data()
    data_loader = tokenizer.prepare_dataloader(batch_size)
    cell_embeddings_1 = evaluate(model, data_loader)

    cell_embeddings_results[l] = (cell_embeddings_0, cell_embeddings_1)


pert_ligand: GSTP1
stFormer - INFO - number of samples: 1395, 
	 feature length of center cell: 561
	 feature length of niche cells: 977


100%|██████████| 70/70 [00:06<00:00, 10.70it/s]


stFormer - INFO - number of samples: 1395, 
	 feature length of center cell: 561
	 feature length of niche cells: 977


100%|██████████| 70/70 [00:06<00:00, 11.06it/s]


pert_ligand: DLL1
stFormer - INFO - number of samples: 961, 
	 feature length of center cell: 515
	 feature length of niche cells: 977


...

100%|██████████| 60/60 [00:05<00:00, 10.37it/s]


pert_ligand: IL7
stFormer - INFO - number of samples: 345, 
	 feature length of center cell: 340
	 feature length of niche cells: 901


100%|██████████| 18/18 [00:01<00:00,  9.39it/s]


stFormer - INFO - number of samples: 345, 
	 feature length of center cell: 340
	 feature length of niche cells: 901


100%|██████████| 18/18 [00:01<00:00,  9.32it/s]


In [106]:
pcc={}
for l in cell_embeddings_results:
    pcc[l] = []
    cell_embeddings_0, cell_embeddings_1 = cell_embeddings_results[l]
    for i in range(cell_embeddings_0.shape[0]):
        pcc[l].append(np.corrcoef(cell_embeddings_0[i], cell_embeddings_1[i])[0,1])
    pcc[l] = np.median(pcc[l])

In [ ]:
import pickle

pickle.dump(pcc, open(f'pcc.pkl', 'wb'))

In [108]:
sorted_keys = sorted(pcc.keys(), key=lambda k: pcc[k])

print("按值从小到大排序的键值对:")
for key in sorted_keys:
    print(f"{key}: {pcc[key]}")

按值从小到大排序的键值对:
HLA-A: 0.9514307785528449
B2M: 0.9557761910562934
VIM: 0.977393560790292
HLA-DRA: 0.9827954008943257
HLA-DRB1: 0.9832962320396783
IL32: 0.9836035996902485
GC: 0.9846634672589745
APOE: 0.9848349083010526
APOA1: 0.9848575261279858
MIF: 0.9864492243189398
ANXA2: 0.987101083495937
CALM2: 0.9873549813728817
TIMP1: 0.9874421592919961
FABP5: 0.987787075310118
LYZ: 0.9878228074096783
COL1A1: 0.9882454209447216
APOC1: 0.988516572045065
BGN: 0.9885252686458432
S100A4: 0.9885706702465222
HSPG2: 0.9891984363478206
MFAP5: 0.9893129259567873
HSP90B1: 0.9893807559419284
HSP90AA1: 0.9895794098371927
LGALS1: 0.9901130644219321
EPCAM: 0.9904086545635422
MMP12: 0.9904806052790239
TNFRSF14: 0.9908729253412518
COL14A1: 0.9909865434130525
CALM1: 0.9911123191483046
APOD: 0.9911746867217158
CD5L: 0.9917951584676119
IL2: 0.9918581305524414
IGFBP7: 0.9920828258748677
SERPINA1: 0.9921820103273491
BST2: 0.9923179140654201
WIF1: 0.9929011467837227
IL1B: 0.9930465341509086
SLPI: 0.993349906099898
C1QA